# Feature Engineering

### 1. Import and load data

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("epl_final.csv")


### 2. Data Cleaning

In [26]:
# Check for missing values
print(df.isnull().sum())

# Drop rows with missing goals or results
df = df.dropna(subset=['FullTimeHomeGoals', 'FullTimeAwayGoals', 'FullTimeResult'])

# Parse dates and sort chronologically
df['MatchDate'] = pd.to_datetime(df['MatchDate'])
df = df.sort_values('MatchDate').reset_index(drop=True)

# Standardize results column
df['result'] = df['FullTimeResult'].map({'H': 'W', 'D': 'D', 'A': 'L'})

print(df.shape)
print(df[['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 
          'FullTimeHomeGoals', 'FullTimeAwayGoals', 'result']].head(10))

Season               0
MatchDate            0
HomeTeam             0
AwayTeam             0
FullTimeHomeGoals    0
FullTimeAwayGoals    0
FullTimeResult       0
HalfTimeHomeGoals    0
HalfTimeAwayGoals    0
HalfTimeResult       0
HomeShots            0
AwayShots            0
HomeShotsOnTarget    0
AwayShotsOnTarget    0
HomeCorners          0
AwayCorners          0
HomeFouls            0
AwayFouls            0
HomeYellowCards      0
AwayYellowCards      0
HomeRedCards         0
AwayRedCards         0
dtype: int64
(9380, 23)
    Season  MatchDate    HomeTeam       AwayTeam  FullTimeHomeGoals  \
0  2000/01 2000-08-19    Charlton       Man City                  4   
1  2000/01 2000-08-19     Chelsea       West Ham                  4   
2  2000/01 2000-08-19    Coventry  Middlesbrough                  1   
3  2000/01 2000-08-19       Derby    Southampton                  2   
4  2000/01 2000-08-19       Leeds        Everton                  2   
5  2000/01 2000-08-19   Leicester    Aston V

### 3. Feature Engineering

#### 3.1. Season-Level Strength Features

In [27]:
def get_points(result):
    if result == 'W': return 3
    if result == 'D': return 1
    return 0

In [28]:
# Build a match log from each team's perspective

home_log = pd.DataFrame({
    'team': df['HomeTeam'],
    'opponent': df['AwayTeam'],
    'date': df['MatchDate'],
    'season': df['Season'],
    'goals_scored': df['FullTimeHomeGoals'],
    'goals_conceded': df['FullTimeAwayGoals'],
    'result': df['result'],
    'is_home': 1
})

away_log = pd.DataFrame({
    'team': df['AwayTeam'],
    'opponent': df['HomeTeam'],
    'date': df['MatchDate'],
    'season': df['Season'],
    'goals_scored': df['FullTimeAwayGoals'],
    'goals_conceded': df['FullTimeHomeGoals'],
    'result': df['result'].map({'W': 'L', 'D': 'D', 'L': 'W'}),
    'is_home': 0
})

team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
team_log['points'] = team_log['result'].apply(get_points)
team_log['goal_diff'] = team_log['goals_scored'] - team_log['goals_conceded']

In [29]:
team_log = team_log.sort_values(['team', 'season', 'date'])

team_log.head(10)

,team,opponent,date,season,goals_scored,goals_conceded,result,is_home,points,goal_diff
5,Arsenal,Sunderland,2000-08-19,2000/01,0,1,L,0,0,-1
21,Arsenal,Liverpool,2000-08-21,2000/01,2,0,W,1,3,2
40,Arsenal,Charlton,2000-08-26,2000/01,5,3,W,1,3,2
71,Arsenal,Chelsea,2000-09-06,2000/01,2,2,D,0,1,0
84,Arsenal,Bradford,2000-09-09,2000/01,1,1,D,0,1,0
106,Arsenal,Coventry,2000-09-16,2000/01,2,1,W,1,3,1
125,Arsenal,Ipswich,2000-09-23,2000/01,1,1,D,0,1,0
153,Arsenal,Man United,2000-10-01,2000/01,1,0,W,1,3,1
160,Arsenal,Aston Villa,2000-10-14,2000/01,1,0,W,1,3,1
188,Arsenal,West Ham,2000-10-21,2000/01,2,1,W,0,3,1


In [30]:
team_log['ppg_season'] = (
    team_log.groupby(['team', 'season'])['points']      #just take the current season's points
    .apply(lambda x: x.shift(1).expanding().mean())     #shift the points column down by 1 and calculate the expanding mean
    .reset_index(level=[0,1], drop=True)
)

team_log['gd_per_game_season'] = (
    team_log.groupby(['team', 'season'])['goal_diff']
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=[0,1], drop=True)
)

home_log_only = team_log[team_log['is_home'] == 1].copy()
home_log_only['ppg_home_only'] = (
    home_log_only.groupby(['team', 'season'])['points']
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=[0,1], drop=True)
)

away_log_only = team_log[team_log['is_home'] == 0].copy()
away_log_only['ppg_away_only'] = (
    away_log_only.groupby(['team', 'season'])['points']
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=[0,1], drop=True)
)

team_log = team_log.merge(
    home_log_only[['team', 'date', 'ppg_home_only']],
    on=['team', 'date'], how='left'
)
team_log = team_log.merge(
    away_log_only[['team', 'date', 'ppg_away_only']],
    on=['team', 'date'], how='left'
)

# ── Sanity check ────────────────────────────────────────────
print(team_log[['team', 'date', 'season', 'is_home', 'points', 
                'ppg_season', 'gd_per_game_season', 
                'ppg_home_only', 'ppg_away_only']].head(20))

       team       date   season  is_home  points  ppg_season  \
0   Arsenal 2000-08-19  2000/01        0       0         NaN   
1   Arsenal 2000-08-21  2000/01        1       3    0.000000   
2   Arsenal 2000-08-26  2000/01        1       3    1.500000   
3   Arsenal 2000-09-06  2000/01        0       1    2.000000   
4   Arsenal 2000-09-09  2000/01        0       1    1.750000   
5   Arsenal 2000-09-16  2000/01        1       3    1.600000   
6   Arsenal 2000-09-23  2000/01        0       1    1.833333   
7   Arsenal 2000-10-01  2000/01        1       3    1.714286   
8   Arsenal 2000-10-14  2000/01        1       3    1.875000   
9   Arsenal 2000-10-21  2000/01        0       3    2.000000   
10  Arsenal 2000-10-28  2000/01        1       3    2.100000   
11  Arsenal 2000-11-04  2000/01        0       3    2.181818   
12  Arsenal 2000-11-11  2000/01        1       1    2.250000   
13  Arsenal 2000-11-18  2000/01        0       0    2.153846   
14  Arsenal 2000-11-26  2000/01        0

#### 3.2 Recent form - Last 5 matches

In [31]:
team_log['ppg_last5'] = (
    team_log.groupby(['team', 'season'])['points']
    .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    .reset_index(level=[0,1], drop=True)
)

# ── Rolling last 5 goals scored ────────────────────────────
team_log['goals_scored_last5'] = (
    team_log.groupby(['team', 'season'])['goals_scored']
    .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    .reset_index(level=[0,1], drop=True)
)

# ── Rolling last 5 goals conceded ──────────────────────────
team_log['goals_conceded_last5'] = (
    team_log.groupby(['team', 'season'])['goals_conceded']
    .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    .reset_index(level=[0,1], drop=True)
)

# ── Sanity check ────────────────────────────────────────────
print(team_log[['team', 'date', 'points', 'goals_scored', 'goals_conceded',
                'ppg_last5', 'goals_scored_last5', 'goals_conceded_last5']].head(20))

       team       date  points  goals_scored  goals_conceded  ppg_last5  \
0   Arsenal 2000-08-19       0             0               1        NaN   
1   Arsenal 2000-08-21       3             2               0       0.00   
2   Arsenal 2000-08-26       3             5               3       1.50   
3   Arsenal 2000-09-06       1             2               2       2.00   
4   Arsenal 2000-09-09       1             1               1       1.75   
5   Arsenal 2000-09-16       3             2               1       1.60   
6   Arsenal 2000-09-23       1             1               1       2.20   
7   Arsenal 2000-10-01       3             1               0       1.80   
8   Arsenal 2000-10-14       3             1               0       1.80   
9   Arsenal 2000-10-21       3             2               1       2.20   
10  Arsenal 2000-10-28       3             5               0       2.60   
11  Arsenal 2000-11-04       3             1               0       2.60   
12  Arsenal 2000-11-11   

#### 3.3 Head-to-head features

In [32]:
h2h = pd.DataFrame({
    'team': df['HomeTeam'],
    'opponent': df['AwayTeam'],
    'date': df['MatchDate'],
    'result': df['FullTimeResult'].map({'H': 'W', 'D': 'D', 'A': 'L'}),
    'goal_diff': df['FullTimeHomeGoals'] - df['FullTimeAwayGoals']
})

h2h_away = pd.DataFrame({
    'team': df['AwayTeam'],
    'opponent': df['HomeTeam'],
    'date': df['MatchDate'],
    'result': df['FullTimeResult'].map({'H': 'L', 'D': 'D', 'A': 'W'}),
    'goal_diff': df['FullTimeAwayGoals'] - df['FullTimeHomeGoals']
})

h2h_all = pd.concat([h2h, h2h_away]).sort_values('date').reset_index(drop=True)
h2h_all['win'] = (h2h_all['result'] == 'W').astype(int)

h2h_records = []

for _, row in df.iterrows():
    home_team = row['HomeTeam']
    away_team = row['AwayTeam']
    match_date = row['MatchDate']

    # All prior meetings between these two teams before this match
    prior = h2h_all[
        (h2h_all['team'] == home_team) &
        (h2h_all['opponent'] == away_team) &
        (h2h_all['date'] < match_date)
    ]

    if len(prior) == 0:
        # No prior meetings
        h2h_records.append({
            'h2h_home_winrate': np.nan,
            'h2h_home_avg_gd': np.nan
        })
    else:
        h2h_records.append({
            'h2h_home_winrate': prior['win'].mean(),
            'h2h_home_avg_gd': prior['goal_diff'].mean()
        })

h2h_df = pd.DataFrame(h2h_records)
df = pd.concat([df, h2h_df], axis=1)


print(df[['MatchDate', 'HomeTeam', 'AwayTeam', 
          'h2h_home_winrate', 'h2h_home_avg_gd']].head(20))

    MatchDate       HomeTeam       AwayTeam  h2h_home_winrate  h2h_home_avg_gd
0  2000-08-19       Charlton       Man City               NaN              NaN
1  2000-08-19        Chelsea       West Ham               NaN              NaN
2  2000-08-19       Coventry  Middlesbrough               NaN              NaN
3  2000-08-19          Derby    Southampton               NaN              NaN
4  2000-08-19          Leeds        Everton               NaN              NaN
5  2000-08-19      Leicester    Aston Villa               NaN              NaN
6  2000-08-19      Liverpool       Bradford               NaN              NaN
7  2000-08-19     Sunderland        Arsenal               NaN              NaN
8  2000-08-19      Tottenham        Ipswich               NaN              NaN
9  2000-08-20     Man United      Newcastle               NaN              NaN
10 2000-08-21        Arsenal      Liverpool               NaN              NaN
11 2000-08-22       Bradford        Chelsea         

In [33]:
home_features = team_log[team_log['is_home'] == 1][[
    'team', 'date',
    'ppg_season', 'gd_per_game_season', 'ppg_home_only',
    'ppg_last5', 'goals_scored_last5', 'goals_conceded_last5'
]].rename(columns={
    'team': 'HomeTeam',
    'date': 'MatchDate',
    'ppg_season': 'home_ppg_season',
    'gd_per_game_season': 'home_gd_per_game_season',
    'ppg_home_only': 'home_ppg_home_only',
    'ppg_last5': 'home_ppg_last5',
    'goals_scored_last5': 'home_goals_scored_last5',
    'goals_conceded_last5': 'home_goals_conceded_last5'
})

# ── Pull away team features from team_log ──────────────────
away_features = team_log[team_log['is_home'] == 0][[
    'team', 'date',
    'ppg_season', 'gd_per_game_season', 'ppg_away_only',
    'ppg_last5', 'goals_scored_last5', 'goals_conceded_last5'
]].rename(columns={
    'team': 'AwayTeam',
    'date': 'MatchDate',
    'ppg_season': 'away_ppg_season',
    'gd_per_game_season': 'away_gd_per_game_season',
    'ppg_away_only': 'away_ppg_away_only',
    'ppg_last5': 'away_ppg_last5',
    'goals_scored_last5': 'away_goals_scored_last5',
    'goals_conceded_last5': 'away_goals_conceded_last5'
})


df['MatchDate'] = pd.to_datetime(df['MatchDate'])
home_features['MatchDate'] = pd.to_datetime(home_features['MatchDate'])
away_features['MatchDate'] = pd.to_datetime(away_features['MatchDate'])
                                            
df = df.merge(home_features, on=['HomeTeam', 'MatchDate'], how='left')
df = df.merge(away_features, on=['AwayTeam', 'MatchDate'], how='left')

final_cols = [
    # Bookkeeping (not fed into model)
    'Season', 'MatchDate', 'HomeTeam', 'AwayTeam',
    # Home team features
    'home_ppg_season', 'home_gd_per_game_season', 'home_ppg_home_only',
    'home_ppg_last5', 'home_goals_scored_last5', 'home_goals_conceded_last5',
    # Away team features
    'away_ppg_season', 'away_gd_per_game_season', 'away_ppg_away_only',
    'away_ppg_last5', 'away_goals_scored_last5', 'away_goals_conceded_last5',
    # Head to head
    'h2h_home_winrate', 'h2h_home_avg_gd',
    # Target
    'FullTimeResult'
]

df_final = df[final_cols].rename(columns={'FullTimeResult': 'result'})

In [36]:
# Handle NaN
print("NaNs before imputation:")
print(df_final.isnull().sum())

df_final = df_final.fillna(df_final.mean(numeric_only=True))

print("\nNaNs after imputation:")
print(df_final.isnull().sum())

print(df_final.shape)
print(df_final.head())

NaNs before imputation:
Season                       0
MatchDate                    0
HomeTeam                     0
AwayTeam                     0
home_ppg_season              0
home_gd_per_game_season      0
home_ppg_home_only           0
home_ppg_last5               0
home_goals_scored_last5      0
home_goals_conceded_last5    0
away_ppg_season              0
away_gd_per_game_season      0
away_ppg_away_only           0
away_ppg_last5               0
away_goals_scored_last5      0
away_goals_conceded_last5    0
h2h_home_winrate             0
h2h_home_avg_gd              0
result                       0
dtype: int64

NaNs after imputation:
Season                       0
MatchDate                    0
HomeTeam                     0
AwayTeam                     0
home_ppg_season              0
home_gd_per_game_season      0
home_ppg_home_only           0
home_ppg_last5               0
home_goals_scored_last5      0
home_goals_conceded_last5    0
away_ppg_season              0
away_gd_p

In [37]:
df_final.to_csv('epl_features.csv', index=False)
print("Saved to epl_features.csv")

Saved to epl_features.csv
